# 01 — Data Preprocessing
**Gurugram Air Quality Study (2020–2024)**

Merges CPCB ground sensors, MODIS LST, and Sentinel-5P satellite data into a single clean dataset for ML training.

**Output:** `outputs/merged_clean.csv`

In [1]:
import os
import pandas as pd
import numpy as np

BASE = os.getcwd()
DATA = os.path.join(BASE, "..", "Datasets")
OUT  = os.path.join(BASE, "outputs")
os.makedirs(OUT, exist_ok=True)

CPCB_PATH     = os.path.join(DATA, "CPCB_Gurugram.csv")
LST_PATH      = os.path.join(DATA, "LST_Gurugram.csv")
SENTINEL_PATH = os.path.join(DATA, "Sentinel_Gurugram.csv")

print("Paths configured.")

Paths configured.


## Helper Functions

In [2]:
def pm25_to_aqi_category(pm):
    """CPCB PM2.5 AQI breakpoints (µg/m³)"""
    if pd.isna(pm): return np.nan
    if pm <= 30:  return "Good"
    if pm <= 60:  return "Moderate"
    if pm <= 90:  return "Poor"
    if pm <= 120: return "Very Poor"
    return "Severe"

def month_to_season(m):
    if m in [12, 1, 2]:    return "Winter"
    if m in [3, 4, 5]:     return "Summer"
    if m in [6, 7, 8, 9]:  return "Monsoon"
    return "Post-Monsoon"

## 1. Load CPCB Ground Data
Filter to Vikas Sadan (site 146) — the most complete monitoring station.

In [3]:
cpcb = pd.read_csv(CPCB_PATH, low_memory=False)
cpcb.columns = cpcb.columns.str.strip()
cpcb["date"] = pd.to_datetime(cpcb["date"], dayfirst=True, errors="coerce")
cpcb = cpcb.dropna(subset=["date"])

# Keep primary station
vikas = cpcb[cpcb["site number"] == 146].copy()

numeric_cols = [
    "PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO (µg/m³)", "NO2 (µg/m³)",
    "NOx (ppb)", "NH3 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)",
    "Ozone (µg/m³)", "AT (°C)", "RH (%)", "WS (m/s)",
    "WD (deg)", "RF (mm)", "SR (W/mt2)", "BP (mmHg)",
]
existing = [c for c in numeric_cols if c in vikas.columns]
for col in existing:
    vikas[col] = pd.to_numeric(vikas[col], errors="coerce")

cpcb_daily = vikas.groupby("date")[existing].mean().reset_index()
cpcb_daily = cpcb_daily.set_index("date").sort_index()
cpcb_daily = cpcb_daily.ffill(limit=3).reset_index()  # fill ≤3-day gaps

print(f"CPCB daily rows: {len(cpcb_daily)}")
cpcb_daily.head()

CPCB daily rows: 1827


C:\Users\Jain\AppData\Local\Temp\ipykernel_14968\2300052107.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cpcb["date"] = pd.to_datetime(cpcb["date"], dayfirst=True, errors="coerce")


,date,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),SR (W/mt2),BP (mmHg)
0,2020-01-01,305.93,NaN,201.53,53.10,203.70,NaN,17.60,3.98,24.78,NaN,80.48,0.97,175.34,NaN,91.05,791.27
1,2020-01-02,71.69,NaN,67.04,52.42,78.21,NaN,8.70,2.01,47.42,NaN,74.94,0.88,197.32,NaN,113.96,786.09
2,2020-01-03,25.41,NaN,23.90,38.65,37.23,NaN,3.36,1.17,69.87,NaN,75.05,0.89,180.76,NaN,180.87,775.50
3,2020-01-04,26.19,NaN,7.73,20.61,16.92,NaN,8.00,0.48,61.52,NaN,54.61,0.89,242.97,NaN,229.00,769.92
4,2020-01-05,53.15,NaN,3.63,26.53,15.18,NaN,2.62,1.17,111.71,NaN,43.43,0.94,140.23,NaN,239.94,761.00


## 2. Load MODIS LST

In [4]:
lst = pd.read_csv(LST_PATH, low_memory=False)
lst.columns = lst.columns.str.strip()
lst["date"] = pd.to_datetime(lst["date"], dayfirst=True, errors="coerce")
lst["LST_C"] = pd.to_numeric(lst["LST_C"], errors="coerce")
lst = lst.dropna(subset=["date", "LST_C"])
lst = lst.groupby("date")["LST_C"].mean().reset_index()

print(f"LST rows: {len(lst)}")
lst.describe()

C:\Users\Jain\AppData\Local\Temp\ipykernel_14968\353962720.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  lst["date"] = pd.to_datetime(lst["date"], dayfirst=True, errors="coerce")


LST rows: 1054


,date,LST_C
count,1054,1054.000000
mean,2021-12-08 05:48:23.225806336,30.112987
min,2020-01-01 00:00:00,-5.550000
25%,2020-12-04 06:00:00,23.844465
50%,2021-12-02 00:00:00,30.585632
75%,2022-12-17 18:00:00,36.345403
max,2023-12-26 00:00:00,51.726491
std,NaN,8.316643


## 3. Load Sentinel-5P
CH4 (entirely empty) and SO2 (mostly empty) are dropped.

In [5]:
sent = pd.read_csv(SENTINEL_PATH, low_memory=False)
sent.columns = sent.columns.str.strip()
sent["date"] = pd.to_datetime(sent["date"], dayfirst=True, errors="coerce")
sent = sent.dropna(subset=["date"])

sent_cols = ["AAI", "CO", "HCHO", "NO2", "O3"]
sent_cols = [c for c in sent_cols if c in sent.columns]
for col in sent_cols:
    sent[col] = pd.to_numeric(sent[col], errors="coerce")

sent = sent[["date"] + sent_cols].groupby("date").mean().reset_index()

print(f"Sentinel rows: {len(sent)}")
sent.head()

C:\Users\Jain\AppData\Local\Temp\ipykernel_14968\1734477605.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sent["date"] = pd.to_datetime(sent["date"], dayfirst=True, errors="coerce")


Sentinel rows: 1800


,date,AAI,CO,HCHO,NO2,O3
0,2020-01-01,-0.816184,0.049067,0.000211,0.000051,0.160842
1,2020-01-02,-1.296559,0.046590,0.000233,0.000170,0.161014
2,2020-01-03,-1.764054,0.041451,0.000201,0.000091,0.142446
3,2020-01-04,-1.539855,0.042510,0.000203,0.000081,0.134150
4,2020-01-05,-0.918930,0.043397,0.000141,0.000067,0.125270


## 4. Merge All Three Datasets

In [6]:
df = cpcb_daily.merge(lst,  on="date", how="left")
df = df.merge(sent, on="date", how="left")
df = df.sort_values("date").reset_index(drop=True)

print(f"Merged shape: {df.shape}")
df.head()

Merged shape: (1827, 23)


,date,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,WD (deg),RF (mm),SR (W/mt2),BP (mmHg),LST_C,AAI,CO,HCHO,NO2,O3
0,2020-01-01,305.93,NaN,201.53,53.10,203.70,NaN,17.60,3.98,24.78,...,175.34,NaN,91.05,791.27,15.271327,-0.816184,0.049067,0.000211,0.000051,0.160842
1,2020-01-02,71.69,NaN,67.04,52.42,78.21,NaN,8.70,2.01,47.42,...,197.32,NaN,113.96,786.09,NaN,-1.296559,0.046590,0.000233,0.000170,0.161014
2,2020-01-03,25.41,NaN,23.90,38.65,37.23,NaN,3.36,1.17,69.87,...,180.76,NaN,180.87,775.50,18.191039,-1.764054,0.041451,0.000201,0.000091,0.142446
3,2020-01-04,26.19,NaN,7.73,20.61,16.92,NaN,8.00,0.48,61.52,...,242.97,NaN,229.00,769.92,NaN,-1.539855,0.042510,0.000203,0.000081,0.134150
4,2020-01-05,53.15,NaN,3.63,26.53,15.18,NaN,2.62,1.17,111.71,...,140.23,NaN,239.94,761.00,18.379494,-0.918930,0.043397,0.000141,0.000067,0.125270


## 5. Feature Engineering

In [7]:
df["month"]       = df["date"].dt.month
df["day_of_year"] = df["date"].dt.dayofyear
df["year"]        = df["date"].dt.year
df["season"]      = df["month"].apply(month_to_season)

season_map = {"Winter": 0, "Summer": 1, "Monsoon": 2, "Post-Monsoon": 3}
df["season_enc"] = df["season"].map(season_map)

pm_col = "PM2.5 (µg/m³)"
df["PM2.5_lag1"] = df[pm_col].shift(1)
df["PM2.5_lag7"] = df[pm_col].shift(7)

df["AQI_category"] = df[pm_col].apply(pm25_to_aqi_category)
aqi_map = {"Good": 0, "Moderate": 1, "Poor": 2, "Very Poor": 3, "Severe": 4}
df["AQI_label"] = df["AQI_category"].map(aqi_map)

# Drop rows with no PM2.5 target
df = df.dropna(subset=[pm_col])
print(f"Final rows after target filter: {len(df)}")

Final rows after target filter: 1768


## 6. Summary Statistics

In [8]:
print("── Null counts ──")
nulls = df.isnull().sum()
print(nulls[nulls > 0])

print("\n── PM2.5 stats ──")
print(df[pm_col].describe().round(2))

print("\n── AQI category distribution ──")
print(df["AQI_category"].value_counts())

── Null counts ──
PM10 (µg/m³)    1753
NH3 (µg/m³)     1768
AT (°C)         1768
RH (%)             9
RF (mm)          371
BP (mmHg)          5
LST_C            742
AAI               34
CO               114
HCHO             122
NO2              150
O3                36
PM2.5_lag1        11
PM2.5_lag7        40
dtype: int64

── PM2.5 stats ──
count    1768.00
mean       89.30
std        57.33
min        10.07
25%        46.42
50%        74.47
75%       117.11
max       450.83
Name: PM2.5 (µg/m³), dtype: float64

── AQI category distribution ──
AQI_category
Moderate     581
Severe       417
Poor         372
Very Poor    286
Good         112
Name: count, dtype: int64


## 7. Save Merged Dataset

In [9]:
out_path = os.path.join(OUT, "merged_clean.csv")
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Saved → f:\Continuing IIPR Reseach Paper\Code\ML_Codes\outputs\merged_clean.csv
Shape: (1768, 32)
Columns: ['date', 'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 'NOx (ppb)', 'NH3 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)', 'AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'RF (mm)', 'SR (W/mt2)', 'BP (mmHg)', 'LST_C', 'AAI', 'CO', 'HCHO', 'NO2', 'O3', 'month', 'day_of_year', 'year', 'season', 'season_enc', 'PM2.5_lag1', 'PM2.5_lag7', 'AQI_category', 'AQI_label']
